# Playing-card classification: corrected evaluation draft

This revision preserves the original 52-class CNN and changes batch handling so training, validation, and test evaluation include the final partial batch. It has not been run: the original `cards.zip` dataset is needed in Google Drive. Saved outputs were cleared to prevent confusing old results with a new run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/drive/My\ Drive/cards.zip -d /content/cards_data/

In [ ]:
!ls /content/cards_data/

In [ ]:
!ls /content/cards_data/cards/train/

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os

In [ ]:
train_dir = "/content/cards_data/cards/train/"
test_dir = "/content/cards_data/cards/test/"
valid_dir = "/content/cards_data/cards/valid/"

In [ ]:
!ls /content/cards_data/cards/train/
!ls /content/cards_data/cards/test/
!ls /content/cards_data/cards/valid/

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
# Image size (resize all images to this size)
img_size = (150, 150)

# Rescale pixel values to [0, 1]
train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load data from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

valid_generator = test_datagen.flow_from_directory(
    valid_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Define the model architecture
model = models.Sequential([
    # Convolutional layers
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    # Flatten the output of convolutional layers
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(512, activation='relu'),

    # Output layer
    layers.Dense(len(train_generator.class_indices), activation='softmax')  # Number of classes
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Let Keras infer the number of batches so each epoch includes the final partial batch.
history = model.fit(train_generator, epochs=10, validation_data=valid_generator)


In [ ]:
# Evaluate all test batches, including the final partial batch.
test_generator.reset()
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f'Test images: {test_generator.samples}')
print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
# Plot accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy over Epochs')
plt.show()

# Plot loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss over Epochs')
plt.show()

The original notebook added dropout after training and evaluation. This revision leaves the architecture unchanged, so a rerun measures the original model with corrected batch handling.
